# DBSCAN
The 6-point example, the effect of eps and min_samples, DBSCAN against k-means, and the weak spots.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from sklearn import datasets
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

## Why k-means is pulled by outliers

In [ ]:
# Nine points around (0, 0) and one outlier at (20, 20): the mean moves to (2, 2)
pts = np.array([[-1, -1], [-1, 0], [-1, 1], [0, -1], [0, 0], [0, 1], [1, -1], [1, 0], [1, 1]])
print("mean without the outlier:", pts.mean(axis=0))
print("mean with the outlier:   ", np.vstack([pts, [20, 20]]).mean(axis=0))

## The 6-point example

In [ ]:
X = np.array([[1, 2], [2, 2], [2, 3], [8, 7], [8, 8], [25, 80]])
for eps, min_samples in [(3, 2), (3, 3), (10, 3)]:
    db = DBSCAN(eps=eps, min_samples=min_samples).fit(X)
    # labels_: one cluster number per point, -1 = noise; core_sample_indices_: which points are core points
    print(f"eps={eps}, min_samples={min_samples}: labels {db.labels_}, core points {db.core_sample_indices_}")

In [ ]:
# There is no predict method: DBSCAN only labels the data it was fitted on
print(hasattr(DBSCAN(), "predict"), hasattr(DBSCAN(), "fit_predict"))

## DBSCAN against k-means

In [ ]:
data = {"moons": datasets.make_moons(500, noise=0.05, random_state=170),
        "circles": datasets.make_circles(500, factor=0.5, noise=0.05, random_state=170)}
for name, (Xd, y) in data.items():
    Xd = StandardScaler().fit_transform(Xd)
    km = KMeans(n_clusters=2, random_state=0).fit_predict(Xd)
    db = DBSCAN(eps=0.3, min_samples=5).fit_predict(Xd)
    print(f"{name}: agreement with the true groups, k-means {adjusted_rand_score(y, km):.2f}, "
          f"DBSCAN {adjusted_rand_score(y, db):.2f}")
fig = go.Figure()
for k in sorted(set(db)):
    fig.add_trace(go.Scatter(x=Xd[db == k, 0], y=Xd[db == k, 1], mode="markers", name=str(k)))
fig.update_layout(template="simple_white", title="DBSCAN on the circles")

## Clusters of different density

In [ ]:
Xv, _ = datasets.make_blobs(n_samples=[300, 100], centers=[(0, 0), (4, 0)], cluster_std=[0.3, 1.3], random_state=0)
for eps in [0.3, 0.5, 0.8]:
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(Xv)
    print(f"eps={eps}: {len(set(lab) - {-1})} clusters, {(lab == -1).sum()} noise points")

## Choosing eps: the k-distance plot

In [ ]:
Xm = StandardScaler().fit_transform(data["moons"][0])
dist, _ = NearestNeighbors(n_neighbors=5).fit(Xm).kneighbors(Xm)   # first neighbour is the point itself
kd = np.sort(dist[:, -1])                                           # distance to the 5th nearest point
go.Figure(go.Scatter(y=kd, mode="lines")).update_layout(template="simple_white",
                                                        yaxis_title="distance to 5th nearest point")

## Playground
The Dash app in `app.py` has a dataset menu and sliders for eps and min_samples. Run `python app.py` in this
folder and open http://127.0.0.1:8050, or set `RUN_APP = True` to show it here.

In [ ]:
RUN_APP = False
if RUN_APP:
    from app import app
    app.run(jupyter_mode="inline")